In [1]:
from dynagraphic.graph import GdcBioGraph
from biodictionary import BioDictionary
from dash import Dash, html, dcc, Output, Input, State, callback, callback_context
import dash_cytoscape as cyto
from datetime import datetime, timedelta

import json

In [2]:
cyto.load_extra_layouts()

In [3]:
biodictionary = BioDictionary()
bg = GdcBioGraph.from_biodictionary(biodictionary)

In [4]:
# layout = {
#     'name': 'klay',
#     'klay': {
#         'direction': 'UP',
#         'layoutHierarchy': True,
#         'spacing': 40
#     }
# }
layout = {
    'name': 'cola',
    'flow': {
        'axis': 'x',
        'minSeparation': -30
    }
}
arrow_color = '#C9C9C9'
base_stylesheet=[
    {
        'selector': 'node',
        'style': {
            'content': 'data(label)'
        }
    },
    {
        'selector': 'edge',
        'style': {
            'curve-style': 'bezier',
            'target-arrow-shape': 'triangle',
            'line-color': arrow_color,
            'target-arrow-color': arrow_color,
        }
    },
    {
        'selector': '.administrative',
        'style': {
            'background-color': '#ffb703'
        }
    },
    {
        'selector': '.biospecimen',
        'style': {
            'background-color': '#8ecae6'
        }
    },
    {
        'selector': '.analysis',
        'style': {
            'background-color': '#fb8500'
        }
    },
    {
        'selector': '.data_file',
        'style': {
            'background-color': '#023047'
        }
    },
    {
        'selector': '.notation',
        'style': {
            'background-color': '#219ebc'
        }
    },
    {
        'selector': 'node.standard',
        'style': {
            'background-opacity': 1.0,
            'border-width': '0px'
        }
    },
    {
        'selector': 'node.fringe',
        'style': {
            'border-width': '3px',
            'border-style': 'dashed',
            'border-opacity': 0.8,
            'border-color': '#C9C9C9',
            'background-opacity': 0.1,
            'text-opacity': 0.1
        }
    },
    {
        'selector': 'edge.standard',
        'style': {

            'line-style': 'solid',
            'line-opacity': 1.0,
            'target-arrow-opacity': 1.0
        }
    },
    {
        'selector': 'edge.fringe',
        'style': {
            'line-style': 'dashed',
            'line-opacity': 0.8,
            'target-arrow-opacity': 0.8
        }
    }
]

In [6]:
app = Dash()

# initialize displayed graph elements
initial_nodes = [
    'program',
    'project',
    'case',
    'aliquot',
    'read_group',
]
bg.set_display_nodes(initial_nodes)
elements = bg.get_cytoscape_elements()

# App layout
controls_bar = html.Div([
    html.Button('Delete Selected Nodes', id='delete-selected-nodes', n_clicks=0)
])
details_view = html.Div(
    [
        html.P("Node Details")
    ],
    style={
        'border-width': '2px',
        'border-color': 'red',
        'border-style': 'solid',
        'width': '400px',
        'height': '100%'
    }
)
cytodiv = cyto.Cytoscape(
    id='biograph',
    elements=elements,
    layout=layout,
    style={'width': '100%', 'height': '600px'},
    stylesheet=base_stylesheet,
    autoRefreshLayout=True
)
left_panel = html.Div([controls_bar, cytodiv], style={'width': '100%', 'height': '100%'})
app_div = html.Div([left_panel, details_view], style={'display': 'flex'})
debug_div = html.Div(html.Pre('Debug Area', id='debug'), style={'background-color': '#F2FAF8'})
app.layout = [app_div, debug_div]
# buffer_time prevents repeated graph state updates
last_interact = datetime.now()
buffer_time = timedelta(seconds=0.1)

@callback(
    Output('biograph', 'elements'),
    Input('biograph', 'tapNode'),
    State('biograph', 'elements')
    )
def handle_tap(node, elements):
    if not node:
        return elements

    trigger_time = datetime.fromtimestamp(node['timeStamp'] / 1000)
    if trigger_time - last_interact < buffer_time:
        return elements

    if node:
        node_id = node['data']['id']
    
        if bg.node_in_subgraph(node_id):
            # node is already in subgraph -> toggle fringe nodes
            bg.toggle_fringes_of_node(node_id)
        else:
            # add node
            bg.add_display_node(node_id)
    
        return bg.get_cytoscape_elements()

# Select one node to perform actions
#  - show fringe nodes when one node is selected
#  - enable delete node button
#  - Node details display
#  - Set properties


# @callback(Output('debug', 'children'), Input('biograph', 'tapNode'))
# def notes(node):
#     if node:
#         datastr = json.dumps(node, indent=3)
#         return str(datastr)

@callback(Output('debug', 'children'), Input('biograph', 'tapEdge'))
def notes(edge):
    if edge:
        datastr = json.dumps(edge, indent=3)
        return str(datastr)


if __name__ == '__main__':
    app.run(debug=True)

Internally the `GdcBioGraph.graph` dictates the possible nodes and hierarchical connectivity. The visual interface enables the user to build a subgraph. The subgraph is represented by the current set of nodes with `display_state` set to `'standard'`.

User interactions can:
- add a node to the subgraph
- remove a node from the subgraph

State update methods will:
- validate addition/subtraction is possible
- update the set of fringe nodes

Node states:
- display: *states describing visual representation*
  - `standard` - visible with node type colors
  - `fringe` - diminished appearance dashed lines pale or no color
  - `hidden` - not visible
- tree *states describing location on graph*
    - `internal` - internal to tree; a root or branch
    - `leaf` - node at edge with no child nodes
    - `detached` - not connected to tree

Standard nodes:
 - display state == `standard`
 - tree state one of `internal`, `leaf`
Fringe nodes:
 - display state == `fringe`
 - tree state == `detached`
Hidden nodes:
 - display state == `hidden`
 - tree state == `detached`

## Potential Applications

- Easily make static images for documentation
- Construct queries for repl
- Programatically construct runs files for a particular data flow and data characteristics
- Visual UI for querying data
- Monitor job queue with context
- Data explorer -- various enumerations
- repl counter, maybe a sanky diagram to track flow and counts
- generate tsv from queries
- development aide for new node schemas